In [ ]:
import os

for root_name, root_path in [
    ("Drishti-GS", "Drishti-GS1_files"),
    ("DRIONS-DB", "DRIONS-DB"),
]:
    print(f"\n=== {root_name} ===")
    for dirpath, dirnames, filenames in os.walk(root_path):
        depth = dirpath.replace(root_path, "").count(os.sep)
        if depth > 2:
            continue
        print(dirpath)
        for f in sorted(filenames)[:8]:
            print("   ", f)
        if len(filenames) > 8:
            print(f"    ... ({len(filenames)} files total)")

In [ ]:
# 1. Peek at one DRIONS-DB expert annotation file
with open(r"DRIONS-DB\DRIONS-DB\experts_anotation\anotExpert1_001.txt") as f:
    lines = f.readlines()
print("Expert1 file, first 10 lines:")
print("".join(lines[:10]))
print(f"...total lines: {len(lines)}")

with open(r"DRIONS-DB\DRIONS-DB\experts_anotation\anotExpert2_001.txt") as f:
    lines2 = f.readlines()
print("\nExpert2 file, first 5 lines:")
print("".join(lines2[:5]))

# 2. Drishti-GS deeper structure
import os
root = r"Drishti-GS1_files\Drishti-GS1_files\Drishti-GS1_files"
for dirpath, dirnames, filenames in os.walk(root):
    depth = dirpath.replace(root, "").count(os.sep)
    if depth > 3:
        continue
    print(dirpath)
    for f in sorted(filenames)[:8]:
        print("   ", f)
    if len(filenames) > 8:
        print(f"    ... ({len(filenames)} files total)")

In [ ]:
import os

gt_folder = r"Drishti-GS1_files\Drishti-GS1_files\Drishti-GS1_files\Training\GT\drishtiGS_002"

for dirpath, dirnames, filenames in os.walk(gt_folder):
    print(dirpath)
    for f in sorted(filenames):
        print("   ", f)

# Also peek at the cdrValues file itself
cdr_path = os.path.join(gt_folder, "drishtiGS_002_cdrValues.txt")
with open(cdr_path) as f:
    print("\n--- cdrValues.txt contents ---")
    print(f.read())

In [ ]:
import os

root = r"RIGA"  # adjust to wherever you extracted it

for dirpath, dirnames, filenames in os.walk(root):
    depth = dirpath.replace(root, "").count(os.sep)
    if depth > 2:
        continue
    print(dirpath)
    for f in sorted(filenames)[:8]:
        print("   ", f)
    if len(filenames) > 8:
        print(f"    ... ({len(filenames)} files total)")

In [ ]:
import os

# 1. See what's actually inside BinRushed / BinRushedcorrected / Magrabia
for sub in [r"RIGA\BinRushed\BinRushed", r"RIGA\BinRushedcorrected\BinRushed", r"RIGA\Magrabia\Magrabia"]:
    print(f"\n=== {sub} ===")
    files = sorted(os.listdir(sub))
    print(f"Total files: {len(files)}")
    print(files[:15])

# 2. Look closely at one full image set from MESSIDOR (image1, all 6 raters + original if present)
mess_dir = r"RIGA\MESSIDOR\MESSIDOR"
image1_files = sorted([f for f in os.listdir(mess_dir) if f.startswith("image1-") or f == "image1.tif"])
print("\n=== image1 full set ===")
print(image1_files)

In [ ]:
import os

for sub in [
    r"RIGA\BinRushed\BinRushed\BinRushed1",
    r"RIGA\BinRushed\BinRushed\BinRushed2",
    r"RIGA\Magrabia\Magrabia\MagrabiaMale",
]:
    print(f"\n=== {sub} ===")
    files = sorted(os.listdir(sub))
    print(f"Total: {len(files)}")
    print(files[:15])

In [ ]:
from PIL import Image
import os

mess_dir = r"RIGA\MESSIDOR\MESSIDOR"
for i in range(1, 7):
    fname = f"image1-{i}.tif"
    img = Image.open(os.path.join(mess_dir, fname))
    print(fname, img.size, img.mode)
    img.convert("RGB").save(f"preview_image1-{i}.png")

In [ ]:
import os
mess_dir = r"RIGA\MESSIDOR\MESSIDOR"
matches = sorted([f for f in os.listdir(mess_dir) if f.lower().startswith("image1prime") or f.lower().startswith("image1.")])
print(matches)

# Also check BinRushed1's numbering — you had image1-3.tif but image1-1/2/4/5/6.jpg (mixed extensions, worth noting)
br1 = r"RIGA\BinRushed\BinRushed\BinRushed1"
print(sorted([f for f in os.listdir(br1) if f.startswith("image1-") or f.startswith("image1prime")]))

In [ ]:
import cv2
import numpy as np
from PIL import Image

mess_dir = r"RIGA\MESSIDOR\MESSIDOR"

def load_rgb(path):
    return np.array(Image.open(path).convert("RGB"))

prime = load_rgb(f"{mess_dir}/image1prime.tif") if False else None  # image1 has no prime, see below
# Instead test on a case that HAS a prime, e.g. from BinRushed2 where we saw image10prime.jpg
br2_dir = r"RIGA\BinRushed\BinRushed\BinRushed2"
prime = load_rgb(f"{br2_dir}/image10prime.jpg")
rater1 = load_rgb(f"{br2_dir}/image10-1.jpg")

# Diff
diff = cv2.absdiff(prime, rater1)
diff_gray = cv2.cvtColor(diff, cv2.COLOR_RGB2GRAY)
_, mask = cv2.threshold(diff_gray, 25, 255, cv2.THRESH_BINARY)

# Clean up small noise
kernel = np.ones((3,3), np.uint8)
mask_clean = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

# Find contours with hierarchy (outer = disc, inner = cup)
contours, hierarchy = cv2.findContours(mask_clean, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_NONE)
areas = [cv2.contourArea(c) for c in contours]
print(f"Found {len(contours)} contours, areas: {sorted(areas, reverse=True)[:5]}")

# Visualize
vis = rater1.copy()
cv2.drawContours(vis, contours, -1, (255, 0, 0), 3)
Image.fromarray(vis).save("diff_check.png")
Image.fromarray(mask_clean).save("mask_check.png")

In [ ]:
import numpy as np
print(np.__version__)
import cv2
print(cv2.__version__)

In [ ]:
import cv2
import numpy as np
from PIL import Image

br2_dir = r"RIGA\BinRushed\BinRushed\BinRushed2"

def load_rgb(path):
    return np.array(Image.open(path).convert("RGB"))

prime = load_rgb(f"{br2_dir}/image10prime.jpg")
rater1 = load_rgb(f"{br2_dir}/image10-1.jpg")

diff = cv2.absdiff(prime, rater1)
diff_gray = cv2.cvtColor(diff, cv2.COLOR_RGB2GRAY)
_, mask = cv2.threshold(diff_gray, 25, 255, cv2.THRESH_BINARY)

kernel = np.ones((3,3), np.uint8)
mask_clean = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

contours, hierarchy = cv2.findContours(mask_clean, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_NONE)
areas = [cv2.contourArea(c) for c in contours]
print(f"Found {len(contours)} contours, areas: {sorted(areas, reverse=True)[:5]}")

vis = rater1.copy()
cv2.drawContours(vis, contours, -1, (255, 0, 0), 3)
Image.fromarray(vis).save("diff_check.png")
Image.fromarray(mask_clean).save("mask_check.png")

In [ ]:
import os
import json
import numpy as np

# Where we'll save all processed loader output — separate from raw downloaded data
OUTPUT_DIR = "processed_data"
os.makedirs(f"{OUTPUT_DIR}/RIGA", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/DRIONS-DB", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/Drishti-GS", exist_ok=True)

def save_record(dataset_name, case_id, record):
    """Saves one image's data (image path + all raters' contours) as a JSON file."""
    path = f"{OUTPUT_DIR}/{dataset_name}/{case_id}.json"
    with open(path, "w") as f:
        json.dump(record, f)
    return path

In [ ]:
import cv2

def reconcile_ring_to_centerline(outer_contour, inner_contour, n_angles=360):
    """
    A hand-drawn ring has an outer edge and inner edge (because the pen stroke
    has thickness). This averages them into one single boundary line, by
    walking around a circle from the ring's center and averaging outer/inner
    radius at each angle.
    """
    outer_pts = outer_contour.reshape(-1, 2).astype(float)
    inner_pts = inner_contour.reshape(-1, 2).astype(float)
    
    cx, cy = outer_pts.mean(axis=0)  # rough center of this ring
    
    def radial_profile(pts):
        dx = pts[:, 0] - cx
        dy = pts[:, 1] - cy
        angles = np.arctan2(dy, dx)
        radii = np.sqrt(dx**2 + dy**2)
        order = np.argsort(angles)
        return angles[order], radii[order]
    
    a_out, r_out = radial_profile(outer_pts)
    a_in, r_in = radial_profile(inner_pts)
    
    query_angles = np.linspace(-np.pi, np.pi, n_angles, endpoint=False)
    r_out_interp = np.interp(query_angles, a_out, r_out, period=2*np.pi)
    r_in_interp = np.interp(query_angles, a_in, r_in, period=2*np.pi)
    
    r_center = (r_out_interp + r_in_interp) / 2.0
    x = cx + r_center * np.cos(query_angles)
    y = cy + r_center * np.sin(query_angles)
    return np.stack([x, y], axis=1)


def extract_disc_cup_contours(prime_img, rater_img, diff_thresh=25):
    """
    Given a clean 'prime' image and one rater's marked-up image,
    returns (disc_contour, cup_contour) as Nx2 point arrays, or (None, None)
    if extraction failed (e.g. couldn't find 2 clean rings).
    """
    diff = cv2.absdiff(prime_img, rater_img)
    diff_gray = cv2.cvtColor(diff, cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(diff_gray, diff_thresh, 255, cv2.THRESH_BINARY)
    kernel = np.ones((3, 3), np.uint8)
    mask_clean = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, hierarchy = cv2.findContours(mask_clean, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_NONE)
    if hierarchy is None or len(contours) < 4:
        return None, None  # couldn't find two clean rings

    hierarchy = hierarchy[0]  # shape (N, 4): [next, prev, first_child, parent]

    # Top-level contours (parent == -1) are outer ring edges; their child is the inner edge
    rings = []
    for i, h in enumerate(hierarchy):
        if h[3] == -1:  # no parent -> this is an outer boundary
            child_idx = h[2]
            if child_idx != -1:
                outer = contours[i]
                inner = contours[child_idx]
                area = cv2.contourArea(outer)
                rings.append((area, outer, inner))

    if len(rings) != 2:
        return None, None  # expected exactly 2 rings (disc, cup)

    rings.sort(key=lambda r: r[0], reverse=True)  # bigger area = disc, smaller = cup
    disc_area, disc_outer, disc_inner = rings[0]
    cup_area, cup_outer, cup_inner = rings[1]

    disc_contour = reconcile_ring_to_centerline(disc_outer, disc_inner)
    cup_contour = reconcile_ring_to_centerline(cup_outer, cup_inner)
    return disc_contour, cup_contour

In [ ]:
from PIL import Image

br2_dir = r"RIGA\BinRushed\BinRushed\BinRushed2"

def load_rgb(path):
    return np.array(Image.open(path).convert("RGB"))

prime = load_rgb(f"{br2_dir}/image10prime.jpg")
rater1 = load_rgb(f"{br2_dir}/image10-1.jpg")

disc_contour, cup_contour = extract_disc_cup_contours(prime, rater1)

print("Disc contour shape:", disc_contour.shape if disc_contour is not None else None)
print("Cup contour shape:", cup_contour.shape if cup_contour is not None else None)

# Visualize: draw both cleaned contours on the image
vis = rater1.copy()
if disc_contour is not None:
    pts = disc_contour.astype(np.int32).reshape(-1, 1, 2)
    cv2.polylines(vis, [pts], isClosed=True, color=(255, 0, 0), thickness=2)
    pts2 = cup_contour.astype(np.int32).reshape(-1, 1, 2)
    cv2.polylines(vis, [pts2], isClosed=True, color=(0, 255, 255), thickness=2)
Image.fromarray(vis).save("reconciled_check.png")

In [ ]:
import glob
import re

def find_riga_cases(subfolder_path):
    """
    Scans a RIGA subfolder (e.g. BinRushed2) and groups files by case.
    Returns a dict: { "image10": {"prime": path_or_None, 1: path, 2: path, ...} }
    """
    all_files = os.listdir(subfolder_path)
    cases = {}
    
    pattern_rater = re.compile(r"^(image\d+)-(\d)\.(jpg|jpeg|tif|tiff)$", re.IGNORECASE)
    pattern_prime = re.compile(r"^(image\d+)prime\.(jpg|jpeg|tif|tiff)$", re.IGNORECASE)
    
    for fname in all_files:
        m_rater = pattern_rater.match(fname)
        m_prime = pattern_prime.match(fname)
        if m_rater:
            case_id, rater_num = m_rater.group(1), int(m_rater.group(2))
            cases.setdefault(case_id, {})[rater_num] = os.path.join(subfolder_path, fname)
        elif m_prime:
            case_id = m_prime.group(1)
            cases.setdefault(case_id, {})["prime"] = os.path.join(subfolder_path, fname)
    
    return cases


def process_riga_subfolder(subfolder_path, source_name, output_dataset_name="RIGA"):
    """
    Processes one RIGA subfolder (e.g. BinRushed2, MESSIDOR).
    Saves one JSON per case with all available raters' disc/cup contours.
    Returns a summary of successes/failures for review.
    """
    cases = find_riga_cases(subfolder_path)
    summary = {"total": 0, "no_prime": 0, "extraction_failed": {}, "success": 0}
    
    for case_id, files in cases.items():
        summary["total"] += 1
        
        if "prime" not in files:
            summary["no_prime"] += 1
            continue  # skip cases with no clean reference image, for now
        
        prime_img = load_rgb(files["prime"])
        raters_data = {}
        
        for rater_num in range(1, 7):
            if rater_num not in files:
                continue
            rater_img = load_rgb(files[rater_num])
            
            # Images must match size to diff correctly
            if rater_img.shape != prime_img.shape:
                rater_img = cv2.resize(rater_img, (prime_img.shape[1], prime_img.shape[0]))
            
            disc_contour, cup_contour = extract_disc_cup_contours(prime_img, rater_img)
            
            if disc_contour is None:
                summary["extraction_failed"].setdefault(case_id, []).append(rater_num)
                continue
            
            raters_data[str(rater_num)] = {
                "disc_contour": disc_contour.tolist(),
                "cup_contour": cup_contour.tolist()
            }
        
        if len(raters_data) > 0:
            record = {
                "image_path": files["prime"],
                "source": source_name,
                "raters": raters_data
            }
            save_record(output_dataset_name, f"{source_name}_{case_id}", record)
            summary["success"] += 1
    
    return summary

In [ ]:
summary = process_riga_subfolder(
    r"RIGA\BinRushed\BinRushed\BinRushed2",
    source_name="BinRushed2"
)
print(summary)

In [ ]:
def diagnose_failure(subfolder_path, case_id, rater_num):
    """Shows the raw diff mask for a failing case so we can see what went wrong."""
    prime_path = os.path.join(subfolder_path, f"{case_id}prime.jpg")
    rater_path = os.path.join(subfolder_path, f"{case_id}-{rater_num}.jpg")
    
    prime_img = load_rgb(prime_path)
    rater_img = load_rgb(rater_path)
    
    print(f"{case_id} rater{rater_num}: prime shape {prime_img.shape}, rater shape {rater_img.shape}")
    
    if rater_img.shape != prime_img.shape:
        rater_img = cv2.resize(rater_img, (prime_img.shape[1], prime_img.shape[0]))
    
    diff = cv2.absdiff(prime_img, rater_img)
    diff_gray = cv2.cvtColor(diff, cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(diff_gray, 25, 255, cv2.THRESH_BINARY)
    kernel = np.ones((3,3), np.uint8)
    mask_clean = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    
    contours, hierarchy = cv2.findContours(mask_clean, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_NONE)
    print(f"  Found {len(contours)} contours, areas: {sorted([cv2.contourArea(c) for c in contours], reverse=True)[:6]}")
    
    Image.fromarray(mask_clean).save(f"debug_{case_id}_r{rater_num}_mask.png")
    Image.fromarray(rater_img).save(f"debug_{case_id}_r{rater_num}_orig.png")
    return mask_clean

subfolder = r"RIGA\BinRushed\BinRushed\BinRushed2"

# One whole-case failure
diagnose_failure(subfolder, "image14", 1)
# The other whole-case failure  
diagnose_failure(subfolder, "image23", 1)
# One single-rater failure, for comparison
diagnose_failure(subfolder, "image10", 2)

In [ ]:
def extract_disc_cup_contours(prime_img, rater_img, diff_thresh=25, min_area=50):
    diff = cv2.absdiff(prime_img, rater_img)
    diff_gray = cv2.cvtColor(diff, cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(diff_gray, diff_thresh, 255, cv2.THRESH_BINARY)
    kernel = np.ones((3, 3), np.uint8)
    mask_clean = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, hierarchy = cv2.findContours(mask_clean, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_NONE)
    if hierarchy is None:
        return None, None
    hierarchy = hierarchy[0]

    # NEW: ignore tiny noise specks (JPEG artifacts, dust) before doing anything else
    keep = [i for i, c in enumerate(contours) if cv2.contourArea(c) >= min_area]
    if len(keep) == 0:
        return None, None

    # Group surviving contours: each top-level (no parent) contour is a "structure"
    # It may have 0 children (single thin line) or 1 child (thick stroke, ring)
    structures = []  # list of (area, contour_or_averaged_centerline)
    for i in keep:
        if hierarchy[i][3] == -1 or hierarchy[i][3] not in keep:  # top-level among kept
            child_idx = hierarchy[i][2]
            if child_idx in keep:
                # ring: has an inner edge too -> average them
                outer, inner = contours[i], contours[child_idx]
                centerline = reconcile_ring_to_centerline(outer, inner)
                area = cv2.contourArea(outer)
            else:
                # single thin line: already a centerline, just resample it evenly
                pts = contours[i].reshape(-1, 2).astype(float)
                cx, cy = pts.mean(axis=0)
                angles = np.arctan2(pts[:,1]-cy, pts[:,0]-cx)
                radii = np.sqrt((pts[:,0]-cx)**2 + (pts[:,1]-cy)**2)
                order = np.argsort(angles)
                query_angles = np.linspace(-np.pi, np.pi, 360, endpoint=False)
                r_interp = np.interp(query_angles, angles[order], radii[order], period=2*np.pi)
                centerline = np.stack([cx + r_interp*np.cos(query_angles),
                                        cy + r_interp*np.sin(query_angles)], axis=1)
                area = cv2.contourArea(contours[i])
            structures.append((area, centerline))

    if len(structures) < 2:
        return None, None  # still couldn't find two separate structures

    structures.sort(key=lambda s: s[0], reverse=True)
    disc_contour = structures[0][1]  # biggest = disc
    cup_contour = structures[1][1]   # second biggest = cup
    return disc_contour, cup_contour

In [ ]:
br2_dir = r"RIGA\BinRushed\BinRushed\BinRushed2"

for case_id, rater in [("image14", 1), ("image23", 1), ("image10", 2)]:
    prime = load_rgb(f"{br2_dir}/{case_id}prime.jpg")
    rater_img = load_rgb(f"{br2_dir}/{case_id}-{rater}.jpg")
    disc, cup = extract_disc_cup_contours(prime, rater_img)
    print(case_id, rater, "disc:", disc.shape if disc is not None else None,
                          "cup:", cup.shape if cup is not None else None)

In [ ]:
summary = process_riga_subfolder(
    r"RIGA\BinRushed\BinRushed\BinRushed2",
    source_name="BinRushed2"
)
print(summary)

In [ ]:
diagnose_failure(r"RIGA\BinRushed\BinRushed\BinRushed2", "image20", 3)

In [ ]:
def find_riga_cases(subfolder_path):
    all_files = os.listdir(subfolder_path)
    cases = {}
    
    pattern_rater = re.compile(r"^(image\d+)-(\d)\.(jpg|jpeg|tif|tiff)$", re.IGNORECASE)
    pattern_prime = re.compile(r"^(image\d+)prime\.(jpg|jpeg|tif|tiff)$", re.IGNORECASE)
    
    for fname in all_files:
        m_rater = pattern_rater.match(fname)
        m_prime = pattern_prime.match(fname)
        if m_rater:
            case_id, rater_num = m_rater.group(1).lower(), int(m_rater.group(2))  # normalize case
            cases.setdefault(case_id, {})[rater_num] = os.path.join(subfolder_path, fname)
        elif m_prime:
            case_id = m_prime.group(1).lower()
            cases.setdefault(case_id, {})["prime"] = os.path.join(subfolder_path, fname)
    
    return cases

In [ ]:
def process_riga_subfolder(subfolder_path, source_name, output_dataset_name="RIGA"):
    cases = find_riga_cases(subfolder_path)
    summary = {"total": 0, "no_prime": 0, "incomplete_annotation": {}, "success": 0}
    
    for case_id, files in cases.items():
        summary["total"] += 1
        
        if "prime" not in files:
            summary["no_prime"] += 1
            continue
        
        prime_img = load_rgb(files["prime"])
        raters_data = {}
        
        for rater_num in range(1, 7):
            if rater_num not in files:
                continue
            rater_img = load_rgb(files[rater_num])
            if rater_img.shape != prime_img.shape:
                rater_img = cv2.resize(rater_img, (prime_img.shape[1], prime_img.shape[0]))
            
            disc_contour, cup_contour = extract_disc_cup_contours(prime_img, rater_img)
            
            if disc_contour is None:
                summary["incomplete_annotation"].setdefault(case_id, []).append(rater_num)
                continue
            
            raters_data[str(rater_num)] = {
                "disc_contour": disc_contour.tolist(),
                "cup_contour": cup_contour.tolist()
            }
        
        if len(raters_data) > 0:
            record = {
                "image_path": files["prime"],
                "source": source_name,
                "raters": raters_data
            }
            save_record(output_dataset_name, f"{source_name}_{case_id}", record)
            summary["success"] += 1
    
    return summary

In [ ]:
for name, path in [
    ("BinRushed/BinRushed1", r"RIGA\BinRushed\BinRushed\BinRushed1"),
    ("BinRushedcorrected/BinRushed1", r"RIGA\BinRushedcorrected\BinRushed\BinRushed1"),
    ("BinRushedcorrected/BinRushed1-Corrected", r"RIGA\BinRushedcorrected\BinRushed\BinRushed1-Corrected"),
]:
    try:
        files = os.listdir(path)
        print(f"{name}: {len(files)} files")
    except FileNotFoundError:
        print(f"{name}: NOT FOUND")

In [ ]:
import filecmp

# 1. Are BinRushed1 and BinRushedcorrected/BinRushed1 actually identical?
p1 = r"RIGA\BinRushed\BinRushed\BinRushed1"
p2 = r"RIGA\BinRushedcorrected\BinRushed\BinRushed1"

files1 = set(os.listdir(p1))
files2 = set(os.listdir(p2))
print("Same filenames?", files1 == files2)
print("Only in p1:", files1 - files2)
print("Only in p2:", files2 - files1)

# spot check a couple of files for identical content
sample_files = list(files1 & files2)[:3]
for f in sample_files:
    same = filecmp.cmp(os.path.join(p1, f), os.path.join(p2, f), shallow=False)
    print(f"{f}: identical content = {same}")

# 2. What's actually inside BinRushed1-Corrected?
p3 = r"RIGA\BinRushedcorrected\BinRushed\BinRushed1-Corrected"
files3 = sorted(os.listdir(p3))
print("\nBinRushed1-Corrected sample files:", files3[:15])

# Which case IDs appear in Corrected vs original?
def extract_case_ids(files):
    ids = set()
    for f in files:
        m = re.match(r"(image\d+)", f, re.IGNORECASE)
        if m:
            ids.add(m.group(1).lower())
    return ids

ids_original = extract_case_ids(files1)
ids_corrected = extract_case_ids(files3)
print(f"\nCase IDs in original BinRushed1: {len(ids_original)}")
print(f"Case IDs in Corrected: {len(ids_corrected)}")
print(f"Case IDs only in Corrected (new?): {ids_corrected - ids_original}")
print(f"Case IDs in both (overlapping/replaced?): {ids_corrected & ids_original}")

In [ ]:
p1 = r"RIGA\BinRushed\BinRushed\BinRushed1"
p3 = r"RIGA\BinRushedcorrected\BinRushed\BinRushed1-Corrected"

def count_primes(folder):
    return len([f for f in os.listdir(folder) if re.match(r"image\d+prime\.", f, re.IGNORECASE)])

print("Primes in original BinRushed1:", count_primes(p1))
print("Primes in BinRushed1-Corrected:", count_primes(p3))

# confirm rater file count matches too
def count_rater_files(folder):
    return len([f for f in os.listdir(folder) if re.match(r"image\d+-\d\.", f, re.IGNORECASE)])

print("Rater files in original:", count_rater_files(p1))
print("Rater files in Corrected:", count_rater_files(p3))

In [ ]:
# 1. Does BinRushedcorrected have Corrected versions of 2, 3, 4 as well?
print(os.listdir(r"RIGA\BinRushedcorrected\BinRushed"))

# 2. Is there any hint of a MESSIDOR-corrected or Magrabia-corrected anywhere?
for root, dirs, files in os.walk(r"RIGA"):
    for d in dirs:
        if "correct" in d.lower():
            print(os.path.join(root, d))

In [ ]:
def count_primes(folder):
    return len([f for f in os.listdir(folder) if re.match(r"image\d+prime\.", f, re.IGNORECASE)])

paths_to_check = {
    "BinRushed3": r"RIGA\BinRushed\BinRushed\BinRushed3",
    "BinRushed4": r"RIGA\BinRushed\BinRushed\BinRushed4",
    "MagrabiaMale": r"RIGA\Magrabia\Magrabia\MagrabiaMale",
    "MagrabiFemale": r"RIGA\Magrabia\Magrabia\MagrabiFemale",
}

for name, path in paths_to_check.items():
    try:
        n_primes = count_primes(path)
        n_total = len(os.listdir(path))
        print(f"{name}: {n_primes} primes out of {n_total} total files")
    except FileNotFoundError:
        print(f"{name}: NOT FOUND")

In [ ]:
riga_subfolders = [
    ("BinRushed1", r"RIGA\BinRushedcorrected\BinRushed\BinRushed1-Corrected"),
    ("BinRushed2", r"RIGA\BinRushed\BinRushed\BinRushed2"),
    ("BinRushed3", r"RIGA\BinRushed\BinRushed\BinRushed3"),
    ("BinRushed4", r"RIGA\BinRushed\BinRushed\BinRushed4"),
    ("MagrabiaMale", r"RIGA\Magrabia\Magrabia\MagrabiaMale"),
    ("MagrabiFemale", r"RIGA\Magrabia\Magrabia\MagrabiFemale"),
    # MESSIDOR intentionally excluded for now — no prime files available
]

all_summaries = {}
grand_total = {"total": 0, "no_prime": 0, "incomplete_raters": 0, "success": 0}

for source_name, path in riga_subfolders:
    print(f"Processing {source_name}...")
    summary = process_riga_subfolder(path, source_name=source_name)
    all_summaries[source_name] = summary
    
    grand_total["total"] += summary["total"]
    grand_total["no_prime"] += summary["no_prime"]
    grand_total["incomplete_raters"] += sum(len(v) for v in summary["incomplete_annotation"].values())
    grand_total["success"] += summary["success"]
    
    print(f"  -> {summary['success']}/{summary['total']} images succeeded, "
          f"{summary['no_prime']} missing prime, "
          f"{sum(len(v) for v in summary['incomplete_annotation'].values())} incomplete rater annotations")

print("\n=== GRAND TOTAL ===")
print(grand_total)

# Show which specific cases had incomplete annotations, across all sources, for the record
print("\n=== Incomplete annotation details (for your report) ===")
for source_name, summary in all_summaries.items():
    if summary["incomplete_annotation"]:
        print(f"{source_name}: {summary['incomplete_annotation']}")

In [ ]:
import json

# Pick any processed case to verify
with open("processed_data/RIGA/BinRushed2_image10.json") as f:
    record = json.load(f)

print("Image path:", record["image_path"])
print("Raters present:", list(record["raters"].keys()))

# Reload the original image and overlay all raters' disc contours
img = load_rgb(record["image_path"])
vis = img.copy()

colors = [(255,0,0), (0,255,0), (0,0,255), (255,255,0), (255,0,255), (0,255,255)]

for i, (rater_id, data) in enumerate(record["raters"].items()):
    disc_pts = np.array(data["disc_contour"], dtype=np.int32).reshape(-1, 1, 2)
    cup_pts = np.array(data["cup_contour"], dtype=np.int32).reshape(-1, 1, 2)
    color = colors[i % len(colors)]
    cv2.polylines(vis, [disc_pts], isClosed=True, color=color, thickness=2)
    cv2.polylines(vis, [cup_pts], isClosed=True, color=color, thickness=2)

Image.fromarray(vis).save("verify_reload.png")

In [ ]:
def parse_drions_expert_file(filepath):
    """Reads one DRIONS-DB expert annotation file: 'x , y' per line."""
    points = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            x_str, y_str = line.split(",")
            points.append([float(x_str.strip()), float(y_str.strip())])
    return np.array(points)


def process_drions_db(base_path, output_dataset_name="DRIONS-DB"):
    images_dir = os.path.join(base_path, "images")
    experts_dir = os.path.join(base_path, "experts_anotation")
    
    summary = {"total": 0, "success": 0, "missing_expert": {}}
    
    image_files = [f for f in os.listdir(images_dir) if f.startswith("image_") and f.endswith(".jpg")]
    
    for img_file in sorted(image_files):
        summary["total"] += 1
        case_num = img_file.replace("image_", "").replace(".jpg", "")  # e.g. "001"
        
        raters_data = {}
        for expert_num in [1, 2]:
            expert_file = os.path.join(experts_dir, f"anotExpert{expert_num}_{case_num}.txt")
            if not os.path.exists(expert_file):
                summary["missing_expert"].setdefault(case_num, []).append(expert_num)
                continue
            
            contour = parse_drions_expert_file(expert_file)
            raters_data[str(expert_num)] = {
                "disc_contour": contour.tolist(),
                "cup_contour": None  # DRIONS-DB has no cup annotation, per the paper
            }
        
        if len(raters_data) > 0:
            record = {
                "image_path": os.path.join(images_dir, img_file),
                "source": "DRIONS-DB",
                "raters": raters_data
            }
            save_record(output_dataset_name, f"DRIONS_{case_num}", record)
            summary["success"] += 1
    
    return summary


summary = process_drions_db(r"DRIONS-DB\DRIONS-DB")
print(summary)

In [ ]:
with open("processed_data/DRIONS-DB/DRIONS_001.json") as f:
    record = json.load(f)

print("Image path:", record["image_path"])
print("Raters present:", list(record["raters"].keys()))

img = load_rgb(record["image_path"])
vis = img.copy()

colors = [(255, 0, 0), (0, 0, 255)]  # 2 experts only

for i, (rater_id, data) in enumerate(record["raters"].items()):
    pts = np.array(data["disc_contour"], dtype=np.int32).reshape(-1, 1, 2)
    color = colors[i % len(colors)]
    cv2.polylines(vis, [pts], isClosed=True, color=color, thickness=2)

Image.fromarray(vis).save("verify_drions.png")

In [ ]:
mess_dir = r"RIGA\MESSIDOR\MESSIDOR"

# Find every file with "prime" in the name, case-insensitive, anywhere in MESSIDOR
prime_files = [f for f in os.listdir(mess_dir) if "prime" in f.lower()]
print(f"Total prime files found: {len(prime_files)}")
print(prime_files[:20])

# Also get the total case count and compare
all_case_ids = set()
for f in os.listdir(mess_dir):
    m = re.match(r"(image\d+)", f, re.IGNORECASE)
    if m:
        all_case_ids.add(m.group(1).lower())
print(f"\nTotal distinct case IDs in MESSIDOR: {len(all_case_ids)}")

prime_case_ids = set()
for f in prime_files:
    m = re.match(r"(image\d+)prime", f, re.IGNORECASE)
    if m:
        prime_case_ids.add(m.group(1).lower())
print(f"Case IDs that HAVE a prime: {len(prime_case_ids)}")
print(f"Case IDs WITHOUT a prime: {len(all_case_ids) - len(prime_case_ids)}")

In [ ]:
summary_messidor = process_riga_subfolder(
    r"RIGA\MESSIDOR\MESSIDOR",
    source_name="MESSIDOR"
)
print(summary_messidor)

In [ ]:
all_riga_summaries = {
    "BinRushed1": {"total": 50, "success": 50, "incomplete": 1},
    "BinRushed2": {"total": 47, "success": 47, "incomplete": 2},
    "BinRushed3": {"total": 47, "success": 47, "incomplete": 0},
    "BinRushed4": {"total": 51, "success": 51, "incomplete": 0},
    "MagrabiaMale": {"total": 47, "success": 47, "incomplete": 1},
    "MagrabiFemale": {"total": 48, "success": 47, "incomplete": 0},  # 1 no_prime
    "MESSIDOR": {"total": 460, "success": 460, "incomplete": 3},
}

grand_total_images = sum(s["total"] for s in all_riga_summaries.values())
grand_success_images = sum(s["success"] for s in all_riga_summaries.values())
grand_incomplete_raters = sum(s["incomplete"] for s in all_riga_summaries.values())

print(f"RIGA total images across all sub-sources: {grand_total_images}")
print(f"Images with at least one usable rater: {grand_success_images}")
print(f"Total incomplete rater-annotations (excluded): {grand_incomplete_raters}")
print(f"Total possible rater-annotations (images x 6): {grand_total_images * 6}")
print(f"Usable rater-annotations: {grand_total_images * 6 - grand_incomplete_raters - 1}")  # -1 for the missing-prime case

# Also count what's actually sitting on disk, as a hard cross-check
riga_files = os.listdir("processed_data/RIGA")
print(f"\nActual JSON files saved: {len(riga_files)}")

In [ ]:
import glob

def contour_to_bbox(contour_pts):
    """Given an Nx2 array of contour points, return (x_min, y_min, x_max, y_max)."""
    pts = np.array(contour_pts)
    return pts[:,0].min(), pts[:,1].min(), pts[:,0].max(), pts[:,1].max()


def get_averaged_disc_bbox(record):
    """Averages the disc bounding box across all available raters for one image."""
    boxes = []
    for rater_id, data in record["raters"].items():
        if data.get("disc_contour") is None:
            continue
        boxes.append(contour_to_bbox(data["disc_contour"]))
    if len(boxes) == 0:
        return None
    boxes = np.array(boxes)
    return boxes.mean(axis=0)  # (x_min, y_min, x_max, y_max) averaged


def bbox_to_yolo_format(bbox, img_width, img_height):
    """Converts (x_min, y_min, x_max, y_max) in pixels to YOLO's normalized format."""
    x_min, y_min, x_max, y_max = bbox
    x_center = (x_min + x_max) / 2 / img_width
    y_center = (y_min + y_max) / 2 / img_height
    width = (x_max - x_min) / img_width
    height = (y_max - y_min) / img_height
    return x_center, y_center, width, height

In [ ]:
with open("processed_data/RIGA/BinRushed2_image10.json") as f:
    record = json.load(f)

img = load_rgb(record["image_path"])
h, w = img.shape[:2]

bbox = get_averaged_disc_bbox(record)
print("Averaged bbox (pixels):", bbox)

yolo_box = bbox_to_yolo_format(bbox, w, h)
print("YOLO format (normalized):", yolo_box)

# Visualize to sanity check
vis = img.copy()
x_min, y_min, x_max, y_max = bbox.astype(int)
cv2.rectangle(vis, (x_min, y_min), (x_max, y_max), (255, 0, 0), 3)
Image.fromarray(vis).save("verify_bbox.png")

In [ ]:
def generate_yolo_labels(processed_dirs, output_images_dir, output_labels_dir):
    """
    processed_dirs: list of folders under processed_data/ to pull from (e.g. RIGA, DRIONS-DB)
    Copies/links images and writes one .txt label file per image.
    """
    os.makedirs(output_images_dir, exist_ok=True)
    os.makedirs(output_labels_dir, exist_ok=True)
    
    count_success = 0
    count_skipped = 0
    
    for dataset_dir in processed_dirs:
        json_files = glob.glob(f"processed_data/{dataset_dir}/*.json")
        for jf in json_files:
            with open(jf) as f:
                record = json.load(f)
            
            bbox = get_averaged_disc_bbox(record)
            if bbox is None:
                count_skipped += 1
                continue
            
            img_path = record["image_path"]
            if not os.path.exists(img_path):
                count_skipped += 1
                continue
            
            img = load_rgb(img_path)
            h, w = img.shape[:2]
            x_c, y_c, bw, bh = bbox_to_yolo_format(bbox, w, h)
            
            case_name = os.path.splitext(os.path.basename(jf))[0]  # e.g. "BinRushed2_image10"
            
            # Save image as a standard jpg (YOLO tools handle jpg/png best; some originals are .tif)
            out_img_path = os.path.join(output_images_dir, f"{case_name}.jpg")
            Image.fromarray(img).save(out_img_path, quality=95)
            
            # Write YOLO label: class 0 = "optic_disc"
            out_label_path = os.path.join(output_labels_dir, f"{case_name}.txt")
            with open(out_label_path, "w") as lf:
                lf.write(f"0 {x_c:.6f} {y_c:.6f} {bw:.6f} {bh:.6f}\n")
            
            count_success += 1
    
    return count_success, count_skipped


success, skipped = generate_yolo_labels(
    processed_dirs=["RIGA", "DRIONS-DB"],
    output_images_dir="yolo_data/all_images",
    output_labels_dir="yolo_data/all_labels"
)
print(f"Labels generated: {success}, skipped: {skipped}")

In [ ]:
import random
import shutil

random.seed(42)  # so the split is reproducible if we ever need to redo this

all_cases = [os.path.splitext(f)[0] for f in os.listdir("yolo_data/all_labels")]
random.shuffle(all_cases)

split_idx = int(0.85 * len(all_cases))
train_cases = all_cases[:split_idx]
val_cases = all_cases[split_idx:]

print(f"Train: {len(train_cases)}, Val: {len(val_cases)}")

# YOLO's expected folder structure
for split_name, case_list in [("train", train_cases), ("val", val_cases)]:
    img_out = f"yolo_data/images/{split_name}"
    label_out = f"yolo_data/labels/{split_name}"
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(label_out, exist_ok=True)
    
    for case in case_list:
        shutil.copy(f"yolo_data/all_images/{case}.jpg", f"{img_out}/{case}.jpg")
        shutil.copy(f"yolo_data/all_labels/{case}.txt", f"{label_out}/{case}.txt")

print("Done splitting.")

In [ ]:
!pip install ultralytics

In [ ]:
import ultralytics
print(ultralytics.__version__)

In [ ]:
yaml_content = """
path: yolo_data
train: images/train
val: images/val

names:
  0: optic_disc
"""

with open("yolo_data/dataset.yaml", "w") as f:
    f.write(yaml_content)

print("Wrote dataset.yaml")

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data="yolo_data/dataset.yaml",
    epochs=15,        # short test run first
    imgsz=384,        # smaller than 512 for speed; we'll go back to 512 for the real run
    batch=8,
    patience=10,
    project="yolo_runs",
    name="disc_detector_test"
)

In [ ]:
!pip install --upgrade matplotlib

In [ ]:
import matplotlib
print(matplotlib.__version__)

In [ ]:
import numpy as np
print("numpy:", np.__version__)

import cv2
print("opencv:", cv2.__version__)

import matplotlib
print("matplotlib:", matplotlib.__version__)

import torch
print("torch:", torch.__version__)

from ultralytics import YOLO
print("ultralytics imported OK")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data="yolo_data/dataset.yaml",
    epochs=15,
    imgsz=384,
    batch=8,
    patience=10,
    project="yolo_runs",
    name="disc_detector_test"
)

In [ ]:
import glob
matches = glob.glob("**/best.pt", recursive=True)
print(matches)

In [ ]:
from PIL import Image
import os

model = YOLO("runs/detect/yolo_runs/disc_detector_test-2/weights/best.pt")

val_images = os.listdir("yolo_data/images/val")[:5]

for img_name in val_images:
    img_path = f"yolo_data/images/val/{img_name}"
    results = model(img_path)
    
    result_img = results[0].plot()
    Image.fromarray(result_img[..., ::-1]).save(f"pred_{img_name}")
    
    if len(results[0].boxes) > 0:
        box = results[0].boxes[0].xywh[0].tolist()
        print(f"{img_name}: predicted center = ({box[0]:.1f}, {box[1]:.1f})")
    else:
        print(f"{img_name}: NO DETECTION")

In [ ]:
print(os.listdir("."))  # to help you locate the pred_*.png files if needed

In [ ]:
def crop_and_normalize(img, center_x, center_y, disc_diameter, target_size=512):
    """
    Crops a square ROI centered on (center_x, center_y), sized 2.6x the disc
    diameter, resizes to target_size x target_size, then applies CLAHE
    (on green channel) + z-score normalization.
    """
    roi_side = disc_diameter * 2.6
    half = roi_side / 2
    
    x_min = int(center_x - half)
    y_min = int(center_y - half)
    x_max = int(center_x + half)
    y_max = int(center_y + half)
    
    h, w = img.shape[:2]
    
    # Pad the image if the crop would go outside its boundaries
    pad_left = max(0, -x_min)
    pad_top = max(0, -y_min)
    pad_right = max(0, x_max - w)
    pad_bottom = max(0, y_max - h)
    
    if any([pad_left, pad_top, pad_right, pad_bottom]):
        img = cv2.copyMakeBorder(img, pad_top, pad_bottom, pad_left, pad_right,
                                   cv2.BORDER_CONSTANT, value=0)
        x_min += pad_left
        x_max += pad_left
        y_min += pad_top
        y_max += pad_top
    
    crop = img[y_min:y_max, x_min:x_max]
    crop_resized = cv2.resize(crop, (target_size, target_size))
    
    # CLAHE on the green channel (paper's spec: green channel has best vessel/disc contrast)
    green = crop_resized[:, :, 1]
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    green_clahe = clahe.apply(green)
    
    enhanced = crop_resized.copy()
    enhanced[:, :, 1] = green_clahe
    
    # Dataset-agnostic z-score normalization (per-image, per-channel)
    enhanced_float = enhanced.astype(np.float32)
    mean = enhanced_float.mean(axis=(0, 1), keepdims=True)
    std = enhanced_float.std(axis=(0, 1), keepdims=True) + 1e-6
    normalized = (enhanced_float - mean) / std
    
    return normalized, crop_resized  # return both normalized (for the model) and viewable version

In [ ]:
model = YOLO("runs/detect/yolo_runs/disc_detector_test-2/weights/best.pt")

test_path = "processed_data/RIGA/BinRushed2_image10.json"
with open(test_path) as f:
    record = json.load(f)

def load_rgb(path):
    return np.array(Image.open(path).convert("RGB"))

img = load_rgb(record["image_path"])
results = model(record["image_path"])

box = results[0].boxes[0].xywh[0].tolist()
center_x, center_y, box_w, box_h = box
disc_diameter = max(box_w, box_h)

normalized, viewable = crop_and_normalize(img, center_x, center_y, disc_diameter)

print("Normalized stats — mean:", normalized.mean(), "std:", normalized.std())
Image.fromarray(viewable).save("verify_crop.png")

In [ ]:
import os
import json
import re
import glob
import shutil
import random
import cv2
import numpy as np
from PIL import Image
from ultralytics import YOLO

In [ ]:
def process_full_preprocessing(processed_dirs, model, output_dir="preprocessed_data"):
    os.makedirs(output_dir, exist_ok=True)
    
    summary = {"total": 0, "success": 0, "no_detection": [], "error": []}
    
    for dataset_dir in processed_dirs:
        json_files = glob.glob(f"processed_data/{dataset_dir}/*.json")
        out_subdir = os.path.join(output_dir, dataset_dir)
        os.makedirs(out_subdir, exist_ok=True)
        
        for jf in json_files:
            summary["total"] += 1
            case_name = os.path.splitext(os.path.basename(jf))[0]
            
            try:
                with open(jf) as f:
                    record = json.load(f)
                
                img_path = record["image_path"]
                if not os.path.exists(img_path):
                    summary["error"].append(case_name)
                    continue
                
                img = load_rgb(img_path)
                results = model(img_path, verbose=False)
                
                if len(results[0].boxes) == 0:
                    summary["no_detection"].append(case_name)
                    continue
                
                box = results[0].boxes[0].xywh[0].tolist()
                center_x, center_y, box_w, box_h = box
                disc_diameter = max(box_w, box_h)
                
                normalized, viewable = crop_and_normalize(img, center_x, center_y, disc_diameter)
                
                # Save both: normalized array (for model training) and viewable version (for sanity checks)
                np.save(os.path.join(out_subdir, f"{case_name}_normalized.npy"), normalized)
                Image.fromarray(viewable).save(os.path.join(out_subdir, f"{case_name}_viewable.jpg"), quality=90)
                
                summary["success"] += 1
            
            except Exception as e:
                summary["error"].append((case_name, str(e)))
    
    return summary


summary = process_full_preprocessing(["RIGA", "DRIONS-DB"], model)
print(f"Total: {summary['total']}, Success: {summary['success']}")
print(f"No detection: {len(summary['no_detection'])}")
print(f"Errors: {len(summary['error'])}")
if summary['no_detection']:
    print("No-detection cases:", summary['no_detection'][:10])
if summary['error']:
    print("Error cases:", summary['error'][:10])

In [ ]:
import os
import json
import re
import glob
import shutil
import random
import cv2
import numpy as np
from PIL import Image
from ultralytics import YOLO

In [ ]:
def load_rgb(path):
    return np.array(Image.open(path).convert("RGB"))


def crop_and_normalize(img, center_x, center_y, disc_diameter, target_size=512):
    roi_side = disc_diameter * 2.6
    half = roi_side / 2
    
    x_min = int(center_x - half)
    y_min = int(center_y - half)
    x_max = int(center_x + half)
    y_max = int(center_y + half)
    
    h, w = img.shape[:2]
    
    pad_left = max(0, -x_min)
    pad_top = max(0, -y_min)
    pad_right = max(0, x_max - w)
    pad_bottom = max(0, y_max - h)
    
    if any([pad_left, pad_top, pad_right, pad_bottom]):
        img = cv2.copyMakeBorder(img, pad_top, pad_bottom, pad_left, pad_right,
                                   cv2.BORDER_CONSTANT, value=0)
        x_min += pad_left
        x_max += pad_left
        y_min += pad_top
        y_max += pad_top
    
    crop = img[y_min:y_max, x_min:x_max]
    crop_resized = cv2.resize(crop, (target_size, target_size))
    
    green = crop_resized[:, :, 1]
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    green_clahe = clahe.apply(green)
    
    enhanced = crop_resized.copy()
    enhanced[:, :, 1] = green_clahe
    
    enhanced_float = enhanced.astype(np.float32)
    mean = enhanced_float.mean(axis=(0, 1), keepdims=True)
    std = enhanced_float.std(axis=(0, 1), keepdims=True) + 1e-6
    normalized = (enhanced_float - mean) / std
    
    return normalized, crop_resized

In [ ]:
model = YOLO("runs/detect/yolo_runs/disc_detector_test-2/weights/best.pt")

In [ ]:
def process_full_preprocessing_v2(processed_dirs, model, output_dir="preprocessed_data"):
    os.makedirs(output_dir, exist_ok=True)
    summary = {"total": 0, "success": 0, "no_detection": [], "error": []}
    
    for dataset_dir in processed_dirs:
        json_files = glob.glob(f"processed_data/{dataset_dir}/*.json")
        out_subdir = os.path.join(output_dir, dataset_dir)
        os.makedirs(out_subdir, exist_ok=True)
        
        for jf in json_files:
            summary["total"] += 1
            case_name = os.path.splitext(os.path.basename(jf))[0]
            
            try:
                with open(jf) as f:
                    record = json.load(f)
                img_path = record["image_path"]
                if not os.path.exists(img_path):
                    summary["error"].append(case_name)
                    continue
                
                img = load_rgb(img_path)
                results = model(img_path, verbose=False)
                if len(results[0].boxes) == 0:
                    summary["no_detection"].append(case_name)
                    continue
                
                box = results[0].boxes[0].xywh[0].tolist()
                center_x, center_y, box_w, box_h = box
                disc_diameter = max(box_w, box_h)
                
                normalized, viewable = crop_and_normalize(img, center_x, center_y, disc_diameter)
                
                roi_side = disc_diameter * 2.6
                half = roi_side / 2
                x_min = center_x - half
                y_min = center_y - half
                scale = 512 / roi_side
                
                transform = {
                    "x_min": x_min, "y_min": y_min, "scale": scale,
                    "center_x": center_x, "center_y": center_y,
                    "disc_diameter": disc_diameter
                }
                
                np.save(os.path.join(out_subdir, f"{case_name}_normalized.npy"), normalized)
                Image.fromarray(viewable).save(os.path.join(out_subdir, f"{case_name}_viewable.jpg"), quality=90)
                with open(os.path.join(out_subdir, f"{case_name}_transform.json"), "w") as tf:
                    json.dump(transform, tf)
                
                summary["success"] += 1
            except Exception as e:
                summary["error"].append((case_name, str(e)))
    
    return summary


summary = process_full_preprocessing_v2(["RIGA", "DRIONS-DB"], model)
print(summary["total"], summary["success"], len(summary["no_detection"]), len(summary["error"]))

In [ ]:
case_name = "BinRushed2_image10"

with open(f"processed_data/RIGA/{case_name}.json") as f:
    record = json.load(f)
with open(f"preprocessed_data/RIGA/{case_name}_transform.json") as f:
    transform = json.load(f)

viewable = load_rgb(f"preprocessed_data/RIGA/{case_name}_viewable.jpg")

x_min, y_min, scale = transform["x_min"], transform["y_min"], transform["scale"]

def map_to_crop_space(contour_pts, x_min, y_min, scale):
    pts = np.array(contour_pts)
    crop_pts = (pts - [x_min, y_min]) * scale
    return crop_pts

vis = viewable.copy()
rater1_disc = record["raters"]["1"]["disc_contour"]
rater1_cup = record["raters"]["1"]["cup_contour"]

disc_crop = map_to_crop_space(rater1_disc, x_min, y_min, scale).astype(np.int32).reshape(-1, 1, 2)
cup_crop = map_to_crop_space(rater1_cup, x_min, y_min, scale).astype(np.int32).reshape(-1, 1, 2)

cv2.polylines(vis, [disc_crop], isClosed=True, color=(255, 0, 0), thickness=2)
cv2.polylines(vis, [cup_crop], isClosed=True, color=(0, 255, 0), thickness=2)

Image.fromarray(vis).save("verify_transform.png")

In [ ]:
def contour_to_radial(contour_pts, center=(256, 256), n_angles=360):
    """
    Converts a closed contour (Nx2 points) into a radial function r(theta),
    sampled at n_angles evenly spaced angles around the given center.
    Returns (angles, radii) both of length n_angles.
    """
    pts = np.array(contour_pts, dtype=float)
    cx, cy = center
    
    dx = pts[:, 0] - cx
    dy = pts[:, 1] - cy
    point_angles = np.arctan2(dy, dx)
    point_radii = np.sqrt(dx**2 + dy**2)
    
    # Sort by angle so interpolation works correctly
    order = np.argsort(point_angles)
    point_angles = point_angles[order]
    point_radii = point_radii[order]
    
    query_angles = np.linspace(-np.pi, np.pi, n_angles, endpoint=False)
    radii = np.interp(query_angles, point_angles, point_radii, period=2*np.pi)
    
    return query_angles, radii

In [ ]:
def check_star_shaped(contour_pts, center=(256, 256)):
    """
    Checks if a contour is star-shaped about the center by verifying that
    each of the original points has a UNIQUE angle (no two points share
    almost the same angle with very different radii, which would indicate
    the boundary crosses that ray more than once).
    """
    pts = np.array(contour_pts, dtype=float)
    cx, cy = center
    angles = np.arctan2(pts[:,1]-cy, pts[:,0]-cx)
    
    # Sort points by angle, then check if angles are monotonically increasing
    # (a proper star-shaped polygon traversed in order should have angles
    # that increase mostly monotonically, possibly wrapping once)
    order = np.argsort(angles)
    sorted_angles = angles[order]
    
    # Count how many original (in-contour-order) angle steps go "backwards"
    # by more than a small tolerance - a robust heuristic for self-intersection
    diffs = np.diff(np.unwrap(angles))
    backwards_steps = np.sum(diffs < -0.1)  # allow tiny numerical noise
    forwards_wraps = np.sum(np.abs(diffs) > np.pi)  # sanity guard
    
    is_star_shaped = backwards_steps == 0
    return is_star_shaped


case_name = "BinRushed2_image10"
with open(f"processed_data/RIGA/{case_name}.json") as f:
    record = json.load(f)

rater1_disc = record["raters"]["1"]["disc_contour"]

# NOTE: these contours are still in ORIGINAL image coordinates, not crop coordinates!
# We need to map them first using the saved transform, same as our last verification step.
with open(f"preprocessed_data/RIGA/{case_name}_transform.json") as f:
    transform = json.load(f)

def map_to_crop_space(contour_pts, x_min, y_min, scale):
    pts = np.array(contour_pts)
    return (pts - [x_min, y_min]) * scale

disc_crop_space = map_to_crop_space(rater1_disc, transform["x_min"], transform["y_min"], transform["scale"])

is_star = check_star_shaped(disc_crop_space, center=(256, 256))
print("Is star-shaped:", is_star)

angles, radii = contour_to_radial(disc_crop_space, center=(256, 256))
print("Radii shape:", radii.shape)
print("Radii range:", radii.min(), "to", radii.max())

In [ ]:
def measure_star_shape_rate(processed_dirs, preprocessed_dir="preprocessed_data"):
    results = {
        "disc": {"total": 0, "star_shaped": 0, "failures": []},
        "cup": {"total": 0, "star_shaped": 0, "failures": []}
    }
    
    for dataset_dir in processed_dirs:
        json_files = glob.glob(f"processed_data/{dataset_dir}/*.json")
        
        for jf in json_files:
            case_name = os.path.splitext(os.path.basename(jf))[0]
            transform_path = f"{preprocessed_dir}/{dataset_dir}/{case_name}_transform.json"
            
            if not os.path.exists(transform_path):
                continue  # skip if preprocessing failed for this case (shouldn't happen, but safe)
            
            with open(jf) as f:
                record = json.load(f)
            with open(transform_path) as f:
                transform = json.load(f)
            
            x_min, y_min, scale = transform["x_min"], transform["y_min"], transform["scale"]
            
            for rater_id, data in record["raters"].items():
                for structure in ["disc_contour", "cup_contour"]:
                    contour = data.get(structure)
                    if contour is None:
                        continue  # e.g. DRIONS-DB has no cup
                    
                    key = "disc" if structure == "disc_contour" else "cup"
                    results[key]["total"] += 1
                    
                    crop_space = map_to_crop_space(contour, x_min, y_min, scale)
                    is_star = check_star_shaped(crop_space, center=(256, 256))
                    
                    if is_star:
                        results[key]["star_shaped"] += 1
                    else:
                        results[key]["failures"].append(f"{dataset_dir}/{case_name}_rater{rater_id}")
    
    return results


star_results = measure_star_shape_rate(["RIGA", "DRIONS-DB"])

for structure in ["disc", "cup"]:
    total = star_results[structure]["total"]
    star = star_results[structure]["star_shaped"]
    pct = 100 * star / total if total > 0 else 0
    print(f"{structure.upper()}: {star}/{total} star-shaped ({pct:.2f}%)")
    print(f"  Failures (first 10): {star_results[structure]['failures'][:10]}")

In [ ]:
# Look at one flagged DRIONS-DB failure directly
case_name = "DRIONS_001"
with open(f"processed_data/DRIONS-DB/{case_name}.json") as f:
    record = json.load(f)
with open(f"preprocessed_data/DRIONS-DB/{case_name}_transform.json") as f:
    transform = json.load(f)

contour = record["raters"]["1"]["disc_contour"]
print("Number of points in this contour:", len(contour))
print("First point:", contour[0])
print("Last point:", contour[-1])

crop_space = map_to_crop_space(contour, transform["x_min"], transform["y_min"], transform["scale"])

# Look at the actual angle sequence to see where the "backward" jump happens
pts = np.array(crop_space)
cx, cy = 256, 256
angles = np.degrees(np.arctan2(pts[:,1]-cy, pts[:,0]-cx))
print("\nAngles (degrees) in original point order:")
print(np.round(angles, 1))

In [ ]:
def check_star_shaped(contour_pts, center=(256, 256), tolerance=0.15):
    """
    Checks if a contour is star-shaped about the center: as we walk around
    the contour in order, the angle relative to center should move
    consistently in ONE direction (either always increasing or always
    decreasing), wrapping at most once. Any point where it reverses
    direction (beyond a small numerical tolerance) indicates the boundary
    crosses some ray from the center more than once - i.e. NOT star-shaped.
    """
    pts = np.array(contour_pts, dtype=float)
    cx, cy = center
    angles = np.arctan2(pts[:,1]-cy, pts[:,0]-cx)
    
    unwrapped = np.unwrap(angles)  # removes false jumps at the +-pi wraparound
    diffs = np.diff(unwrapped)
    
    # A star-shaped contour walked consistently should have diffs that are
    # ALL roughly the same sign (all positive or all negative).
    # Check the dominant direction, then flag steps that go against it.
    net_direction = np.sign(unwrapped[-1] - unwrapped[0])
    if net_direction == 0:
        net_direction = 1  # degenerate edge case guard
    
    wrong_direction_steps = np.sum(diffs * net_direction < -tolerance)
    
    return wrong_direction_steps == 0

In [ ]:
# Re-test the specific case that failed before
is_star = check_star_shaped(crop_space, center=(256, 256))
print("DRIONS_001 rater1 disc, is star-shaped now:", is_star)

# Re-run the full measurement with the fixed checker
star_results = measure_star_shape_rate(["RIGA", "DRIONS-DB"])

for structure in ["disc", "cup"]:
    total = star_results[structure]["total"]
    star = star_results[structure]["star_shaped"]
    pct = 100 * star / total if total > 0 else 0
    print(f"{structure.upper()}: {star}/{total} star-shaped ({pct:.2f}%)")
    print(f"  Failures (first 10): {star_results[structure]['failures'][:10]}")

In [ ]:
case_str = "RIGA/BinRushed2_image36_rater2"
dataset_dir, rest = case_str.split("/")
case_name, rater_part = rest.rsplit("_rater", 1)

with open(f"processed_data/{dataset_dir}/{case_name}.json") as f:
    record = json.load(f)
with open(f"preprocessed_data/{dataset_dir}/{case_name}_transform.json") as f:
    transform = json.load(f)

viewable = load_rgb(f"preprocessed_data/{dataset_dir}/{case_name}_viewable.jpg")
cup_contour = record["raters"][rater_part]["cup_contour"]
crop_space = map_to_crop_space(cup_contour, transform["x_min"], transform["y_min"], transform["scale"])

vis = viewable.copy()
pts = crop_space.astype(np.int32).reshape(-1, 1, 2)
cv2.polylines(vis, [pts], isClosed=True, color=(255, 0, 0), thickness=2)
Image.fromarray(vis).save("verify_nonstar_case.png")

In [ ]:
def build_harmonic_basis(M, N=360):
    """
    Builds the orthonormal circular-harmonic basis matrix Phi (N x (2M+1)),
    matching the paper's Eq. 2: DC term + cos/sin pairs up to order M.
    """
    theta = np.linspace(0, 2*np.pi, N, endpoint=False)
    columns = [np.ones(N) / np.sqrt(N)]  # DC term
    for m in range(1, M + 1):
        columns.append(np.sqrt(2/N) * np.cos(m * theta))
        columns.append(np.sqrt(2/N) * np.sin(m * theta))
    return np.stack(columns, axis=1)  # shape (N, 2M+1)


def encode_decode(r, M, N=360):
    """
    Projects radial function r onto order-M harmonic basis, then reconstructs.
    Returns r_reconstructed.
    """
    Phi = build_harmonic_basis(M, N)
    u = Phi.T @ r          # harmonic coefficients
    r_hat = Phi @ u         # reconstruction
    return r_hat

In [ ]:
def radial_to_mask(radii, center=(256, 256), img_size=512, n_angles=360):
    """Converts a radial function back into a filled binary mask."""
    theta = np.linspace(0, 2*np.pi, n_angles, endpoint=False)
    cx, cy = center
    x = cx + radii * np.cos(theta)
    y = cy + radii * np.sin(theta)
    pts = np.stack([x, y], axis=1).astype(np.int32).reshape(-1, 1, 2)
    
    mask = np.zeros((img_size, img_size), dtype=np.uint8)
    cv2.fillPoly(mask, [pts], 255)
    return mask


def dice_score(mask1, mask2):
    intersection = np.logical_and(mask1 > 0, mask2 > 0).sum()
    total = (mask1 > 0).sum() + (mask2 > 0).sum()
    if total == 0:
        return 1.0
    return 2 * intersection / total

In [ ]:
case_name = "BinRushed2_image10"
with open(f"processed_data/RIGA/{case_name}.json") as f:
    record = json.load(f)
with open(f"preprocessed_data/RIGA/{case_name}_transform.json") as f:
    transform = json.load(f)

disc_contour = record["raters"]["1"]["disc_contour"]
crop_space = map_to_crop_space(disc_contour, transform["x_min"], transform["y_min"], transform["scale"])

angles, radii_original = contour_to_radial(crop_space, center=(256, 256))
mask_original = radial_to_mask(radii_original)

print("Testing M sweep on one contour:")
for M in [4, 8, 12, 16, 20]:
    radii_reconstructed = encode_decode(radii_original, M)
    mask_reconstructed = radial_to_mask(radii_reconstructed)
    dice = dice_score(mask_original, mask_reconstructed)
    print(f"  M={M}: Dice = {dice:.4f}")

In [ ]:
def run_harmonic_sweep(processed_dirs, M_values=[4, 8, 12, 16, 20], preprocessed_dir="preprocessed_data"):
    results = {M: {"disc": [], "cup": []} for M in M_values}
    
    for dataset_dir in processed_dirs:
        json_files = glob.glob(f"processed_data/{dataset_dir}/*.json")
        
        for jf in json_files:
            case_name = os.path.splitext(os.path.basename(jf))[0]
            transform_path = f"{preprocessed_dir}/{dataset_dir}/{case_name}_transform.json"
            if not os.path.exists(transform_path):
                continue
            
            with open(jf) as f:
                record = json.load(f)
            with open(transform_path) as f:
                transform = json.load(f)
            x_min, y_min, scale = transform["x_min"], transform["y_min"], transform["scale"]
            
            for rater_id, data in record["raters"].items():
                for structure, key in [("disc_contour", "disc"), ("cup_contour", "cup")]:
                    contour = data.get(structure)
                    if contour is None:
                        continue
                    
                    crop_space = map_to_crop_space(contour, x_min, y_min, scale)
                    
                    # Skip non-star-shaped contours (r(theta) is undefined/unreliable for these)
                    if not check_star_shaped(crop_space, center=(256, 256)):
                        continue
                    
                    angles, radii_original = contour_to_radial(crop_space, center=(256, 256))
                    mask_original = radial_to_mask(radii_original)
                    
                    for M in M_values:
                        radii_reconstructed = encode_decode(radii_original, M)
                        mask_reconstructed = radial_to_mask(radii_reconstructed)
                        dice = dice_score(mask_original, mask_reconstructed)
                        results[M][key].append(dice)
    
    return results


sweep_results = run_harmonic_sweep(["RIGA", "DRIONS-DB"])

print(f"{'M':>4} | {'Disc Dice (mean±std)':>22} | {'Cup Dice (mean±std)':>22} | {'N disc':>7} | {'N cup':>6}")
for M in [4, 8, 12, 16, 20]:
    disc_dices = np.array(sweep_results[M]["disc"])
    cup_dices = np.array(sweep_results[M]["cup"])
    print(f"{M:>4} | {disc_dices.mean()*100:>8.2f} ± {disc_dices.std()*100:<8.2f} | "
          f"{cup_dices.mean()*100:>8.2f} ± {cup_dices.std()*100:<8.2f} | "
          f"{len(disc_dices):>7} | {len(cup_dices):>6}")

In [ ]:
def compute_energy_spectrum(processed_dirs, M_max=20, preprocessed_dir="preprocessed_data"):
    """Computes the per-harmonic-order energy, averaged across all disc+cup contours."""
    Phi = build_harmonic_basis(M_max, N=360)
    all_energies = []  # each row: energy per coefficient for one contour
    
    for dataset_dir in processed_dirs:
        json_files = glob.glob(f"processed_data/{dataset_dir}/*.json")
        for jf in json_files:
            case_name = os.path.splitext(os.path.basename(jf))[0]
            transform_path = f"{preprocessed_dir}/{dataset_dir}/{case_name}_transform.json"
            if not os.path.exists(transform_path):
                continue
            with open(jf) as f:
                record = json.load(f)
            with open(transform_path) as f:
                transform = json.load(f)
            x_min, y_min, scale = transform["x_min"], transform["y_min"], transform["scale"]
            
            for rater_id, data in record["raters"].items():
                for structure in ["disc_contour", "cup_contour"]:
                    contour = data.get(structure)
                    if contour is None:
                        continue
                    crop_space = map_to_crop_space(contour, x_min, y_min, scale)
                    if not check_star_shaped(crop_space, center=(256, 256)):
                        continue
                    _, radii = contour_to_radial(crop_space, center=(256, 256))
                    u = Phi.T @ radii  # harmonic coefficients: [DC, cos1, sin1, cos2, sin2, ...]
                    all_energies.append(u**2)
    
    all_energies = np.array(all_energies)
    mean_energy_per_coef = all_energies.mean(axis=0)
    
    # Group coefficients by harmonic order m (DC is order 0, then cos/sin pairs per m)
    energy_per_m = [mean_energy_per_coef[0]]  # DC
    for m in range(1, M_max + 1):
        cos_idx = 1 + 2*(m-1)
        sin_idx = 2 + 2*(m-1)
        energy_per_m.append(mean_energy_per_coef[cos_idx] + mean_energy_per_coef[sin_idx])
    
    energy_per_m = np.array(energy_per_m)
    total_energy = energy_per_m.sum()
    cumulative = np.cumsum(energy_per_m) / total_energy
    
    return energy_per_m, cumulative


energy_per_m, cumulative = compute_energy_spectrum(["RIGA", "DRIONS-DB"])

print("Cumulative energy captured by harmonic order m:")
for m in [2, 4, 8, 12, 16, 20]:
    print(f"  m <= {m}: {cumulative[m]*100:.2f}%")

In [ ]:
def compute_energy_spectrum_v2(processed_dirs, M_max=20, preprocessed_dir="preprocessed_data"):
    Phi = build_harmonic_basis(M_max, N=360)
    all_energies = []
    
    for dataset_dir in processed_dirs:
        json_files = glob.glob(f"processed_data/{dataset_dir}/*.json")
        for jf in json_files:
            case_name = os.path.splitext(os.path.basename(jf))[0]
            transform_path = f"{preprocessed_dir}/{dataset_dir}/{case_name}_transform.json"
            if not os.path.exists(transform_path):
                continue
            with open(jf) as f:
                record = json.load(f)
            with open(transform_path) as f:
                transform = json.load(f)
            x_min, y_min, scale = transform["x_min"], transform["y_min"], transform["scale"]
            
            for rater_id, data in record["raters"].items():
                for structure in ["disc_contour", "cup_contour"]:
                    contour = data.get(structure)
                    if contour is None:
                        continue
                    crop_space = map_to_crop_space(contour, x_min, y_min, scale)
                    if not check_star_shaped(crop_space, center=(256, 256)):
                        continue
                    _, radii = contour_to_radial(crop_space, center=(256, 256))
                    u = Phi.T @ radii
                    all_energies.append(u**2)
    
    all_energies = np.array(all_energies)
    mean_energy_per_coef = all_energies.mean(axis=0)
    
    energy_per_m = [mean_energy_per_coef[0]]  # keep DC separately for reference
    for m in range(1, M_max + 1):
        cos_idx = 1 + 2*(m-1)
        sin_idx = 2 + 2*(m-1)
        energy_per_m.append(mean_energy_per_coef[cos_idx] + mean_energy_per_coef[sin_idx])
    energy_per_m = np.array(energy_per_m)
    
    dc_energy = energy_per_m[0]
    ac_energy = energy_per_m[1:]  # exclude DC
    ac_total = ac_energy.sum()
    ac_cumulative = np.cumsum(ac_energy) / ac_total
    
    print(f"DC (m=0) energy: {dc_energy:.2f} ({100*dc_energy/energy_per_m.sum():.2f}% of total including DC)")
    print(f"AC (m>=1) total energy: {ac_total:.2f}")
    
    return ac_energy, ac_cumulative


ac_energy, ac_cumulative = compute_energy_spectrum_v2(["RIGA", "DRIONS-DB"])

print("\nCumulative AC (shape-only) energy captured by harmonic order m:")
for m in [1, 2, 4, 8, 12, 16, 20]:
    print(f"  m <= {m}: {ac_cumulative[m-1]*100:.2f}%")